# Day 2 — HSBC M1 Scrambled Sobol RQMC and European Greek Validation

## tl;dr

This executed notebook imports the reliable market inputs for HSBC note `40447DKU1 Corp` and validates the production simulation interface before it is used for the autocallable payoff.

- Validation instruments: one-year at-the-money European calls on SPX, NDX and RTY.
- Methods: independent pseudo-random MC replications and independently scrambled Sobol RQMC replications.
- Production ordering: time-major, asset-minor, with correlated increments at each time step.
- Validated quantities: price, Delta, Vega and Gamma against Black–Scholes closed forms.
- At \(N=65{,}536\), the mean RQMC/MC RMSE ratios across the three indices are approximately **0.13 for price, 0.38 for Delta and 0.14 for Vega**.
- Gamma is the exception: the corresponding ratio is approximately **1.13**, so plain time-major RQMC does not provide a stable second-order advantage.
- A 1% Gamma spot bump is retained as the baseline; the 0.25%–2.00% bump study remains visible because second differences are materially noisier.
- Data boundary: the underlying snapshots and histories are usable, and the three official initial reference levels are loaded from the issuer supplement. Note-price history and listed-option quotes remain unavailable but are outside the Day 2 validation scope.

## Context & Methods

This is the **engine-validation layer**, not the final fair-value calculation for the HSBC autocallable. European calls have closed-form prices and Greeks, so they reveal implementation, randomisation and finite-difference errors before autocall, coupon, barrier and worst-of discontinuities are introduced.

### Key assumptions

- Risk-neutral GBM with constant \(r,q,\sigma\).
- \(T=1\) year and strike \(K=S_0\), aligned with the workbook's 12M ATM implied volatilities.
- The workbook contains a 3Y SOFR proxy rather than a 1Y OIS zero rate; it is used transparently for Day 2 engine validation.
- The three European calls are marginal validation instruments. Historical correlation is still applied to exercise the three-asset production path interface.
- Vega is reported per 1.00 absolute volatility; divide by 100 for value per one volatility point.
- Delta uses a 0.10% spot bump, Vega a 0.50 volatility-point bump, and Gamma a 1.00% spot bump, all with common random numbers.

For each method and sample size \(N=2^m\), independent replication-level estimates are retained. RQMC uncertainty is estimated across independent scrambles; Sobol points are not treated as i.i.d. observations.

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import os
import tempfile
import time

import numpy as np
import pandas as pd
import openpyxl
from scipy.stats import norm, qmc

os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "hsbc_day2_mpl_cache")
)
import matplotlib.pyplot as plt

pd.set_option("display.precision", 8)
plt.style.use("seaborn-v0_8-whitegrid")


def find_project_root():
    candidates = []
    override = os.environ.get("AP_PROJECT_ROOT")
    if override:
        candidates.append(Path(override).expanduser())
    candidates.extend([Path.cwd(), *Path.cwd().parents])
    notebook_dir = Path(globals().get("__file__", Path.cwd())).resolve().parent
    candidates.extend([notebook_dir, *notebook_dir.parents])
    for candidate in candidates:
        if (candidate / "config" / "core_project_config.json").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate config/core_project_config.json. "
        "Run this notebook from the project directory or set AP_PROJECT_ROOT."
    )


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest().upper()


PROJECT_DIR = find_project_root()
CONFIG_FILE = PROJECT_DIR / "config" / "core_project_config.json"
with CONFIG_FILE.open(encoding="utf-8") as stream:
    PROJECT_CONFIG = json.load(stream)

SOURCE_FILE = PROJECT_DIR / PROJECT_CONFIG["market_data"]["relative_path"]
assert SOURCE_FILE.is_file(), f"Source workbook not found: {SOURCE_FILE}"
SOURCE_SHA256 = sha256_file(SOURCE_FILE)
EXPECTED_SHA256 = PROJECT_CONFIG["market_data"]["sha256"].upper()
assert SOURCE_SHA256 == EXPECTED_SHA256, (
    f"Workbook hash mismatch: expected {EXPECTED_SHA256}, got {SOURCE_SHA256}"
)

OUTPUT_DIR = PROJECT_DIR / "outputs" / "day2_hsbc_m1_rqmc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BASE_SEED = 20260728
T = 1.0
N_STEPS = 12
SAMPLE_SIZES = [2**10, 2**12, 2**14, 2**16]
N_REPLICATIONS = 16
DELTA_BUMP = 0.001
VEGA_BUMP_ABS = 0.005
GAMMA_BUMP = 0.01
UNIFORM_EPS = np.finfo(float).eps

print(f"Project root: {PROJECT_DIR}")
print(f"Source: {SOURCE_FILE}")
print(f"Verified SHA-256: {SOURCE_SHA256}")
print(f"N grid: {SAMPLE_SIZES}; independent replications/scrambles: {N_REPLICATIONS}")

## Data

### 1. Import and validate the Bloomberg workbook

Cached values are read without recalculating Bloomberg formulas. Sentinel zeros, `#NAME?` cells and blank formula outputs are not treated as market observations.

In [ ]:
def is_number(value):
    return (
        isinstance(value, (int, float, np.integer, np.floating))
        and not isinstance(value, bool)
        and np.isfinite(value)
    )


def read_hsbc_inputs(path):
    wb = openpyxl.load_workbook(path, data_only=True, read_only=False)
    setup = wb["Setup"]
    note_snapshot = wb["Note_Snapshot"]
    underlying_snapshot = wb["Underlying_Snapshot"]
    underlying_history = wb["Underlying_History"]
    market_history = wb["Market_History"]
    market_comparison = wb["Market_Comparison"]
    listed_options = wb["Listed_Options"]
    issuer_initial = wb["Issuer_Initial_Value"]

    tickers = [underlying_snapshot.cell(row, 2).value for row in (6, 7, 8)]
    spots = np.array(
        [underlying_snapshot.cell(row, 4).value for row in (6, 7, 8)],
        dtype=float,
    )
    dividend_yields = np.array(
        [underlying_snapshot.cell(row, 5).value for row in (6, 7, 8)],
        dtype=float,
    ) / 100.0
    volatilities = np.array(
        [underlying_snapshot.cell(row, 8).value for row in (6, 7, 8)],
        dtype=float,
    ) / 100.0
    # Underlying_Snapshot!I6:I8 contains zero placeholders. The issuer
    # supplement is the authoritative source for the contractual initial levels.
    initial_references = np.array(
        [issuer_initial.cell(row, 3).value for row in (38, 39, 40)],
        dtype=float,
    )

    history = []
    for date_col, value_col, ticker in zip((1, 4, 7), (2, 5, 8), tickers):
        rows = {}
        for row in range(6, underlying_history.max_row + 1):
            date_value = underlying_history.cell(row, date_col).value
            level = underlying_history.cell(row, value_col).value
            if hasattr(date_value, "year") and is_number(level) and level > 0:
                rows[pd.Timestamp(date_value)] = float(level)
        history.append(pd.Series(rows, name=ticker).sort_index())
    prices = pd.concat(history, axis=1, join="inner").dropna()
    log_returns = np.log(prices / prices.shift(1)).dropna()
    historical_correlation = log_returns.corr().to_numpy()

    rate_rows = []
    for row in range(6, market_history.max_row + 1):
        date_value = market_history.cell(row, 10).value
        rate_pct = market_history.cell(row, 11).value
        if hasattr(date_value, "year") and is_number(rate_pct) and rate_pct > 0:
            rate_rows.append((pd.Timestamp(date_value), float(rate_pct) / 100.0))
    rate_series = pd.Series(dict(rate_rows), name="3Y SOFR proxy").sort_index()
    risk_free_rate = float(rate_series.iloc[-1])

    note_prices = []
    for row in range(6, market_comparison.max_row + 1):
        value = market_comparison.cell(row, 7).value
        if is_number(value) and value > 0:
            note_prices.append(float(value))

    valid_listed_options = 0
    for row in range(6, listed_options.max_row + 1):
        bid, ask = listed_options.cell(row, 7).value, listed_options.cell(row, 8).value
        if is_number(bid) and is_number(ask) and bid > 0 and ask >= bid:
            valid_listed_options += 1

    metadata = {
        "Bloomberg ID": setup["B6"].value,
        "CUSIP": note_snapshot["D6"].value,
        "ISIN": note_snapshot["E6"].value,
        "Issuer": note_snapshot["B6"].value,
        "Issue date": note_snapshot["F6"].value,
        "Maturity": note_snapshot["G6"].value,
        "As-of date": setup["B8"].value,
        "Setup pricing date": setup["B14"].value,
        "Setup issue date": setup["B15"].value,
        "Setup maturity date": setup["B16"].value,
    }
    return {
        "tickers": tickers,
        "spots": spots,
        "dividend_yields": dividend_yields,
        "volatilities": volatilities,
        "initial_references": initial_references,
        "prices": prices,
        "log_returns": log_returns,
        "correlation": historical_correlation,
        "risk_free_rate": risk_free_rate,
        "rate_series": rate_series,
        "valid_note_prices": len(note_prices),
        "valid_listed_options": valid_listed_options,
        "metadata": metadata,
    }


inputs = read_hsbc_inputs(SOURCE_FILE)
market_inputs = pd.DataFrame(
    {
        "Ticker": inputs["tickers"],
        "Spot": inputs["spots"],
        "Dividend yield": inputs["dividend_yields"],
        "12M ATM volatility": inputs["volatilities"],
        "Initial reference": inputs["initial_references"],
    }
)
display(pd.Series(inputs["metadata"], name="Value").to_frame())
display(market_inputs)
display(
    pd.DataFrame(
        inputs["correlation"],
        index=inputs["tickers"],
        columns=inputs["tickers"],
    )
)
print(
    f"Aligned underlying history: {inputs['prices'].index.min().date()} to "
    f"{inputs['prices'].index.max().date()} ({len(inputs['prices'])} closes)"
)
print(
    f"Risk-free proxy: {inputs['risk_free_rate']:.4%} "
    f"as of {inputs['rate_series'].index[-1].date()}"
)

In [ ]:
eigenvalues = np.linalg.eigvalsh(inputs["correlation"])
pricing_date = pd.Timestamp(inputs["metadata"]["Setup pricing date"])
issue_date = pd.Timestamp(inputs["metadata"]["Setup issue date"])
pricing_to_issue_days = (issue_date - pricing_date).days
quality_rows = [
    {
        "Dataset / check": "Current SPX/NDX/RTY market inputs",
        "Evidence": f"{np.isfinite(inputs['spots']).sum()}/3 spots, "
                    f"{np.isfinite(inputs['dividend_yields']).sum()}/3 yields, "
                    f"{np.isfinite(inputs['volatilities']).sum()}/3 vols",
        "Status": "PASS",
        "Use in Day 2": "Core",
    },
    {
        "Dataset / check": "Aligned underlying closes",
        "Evidence": f"{len(inputs['prices'])} common dates",
        "Status": "PASS" if len(inputs["prices"]) >= 200 else "CHECK",
        "Use in Day 2": "Historical correlation",
    },
    {
        "Dataset / check": "Correlation positive semidefinite",
        "Evidence": f"minimum eigenvalue {eigenvalues.min():.6f}",
        "Status": "PASS" if eigenvalues.min() > -1e-10 else "CHECK",
        "Use in Day 2": "Correlated path interface",
    },
    {
        "Dataset / check": "Positive SOFR proxy",
        "Evidence": f"{inputs['risk_free_rate']:.4%}",
        "Status": "PASS" if inputs["risk_free_rate"] > 0 else "CHECK",
        "Use in Day 2": "Discounting",
    },
    {
        "Dataset / check": "Pricing-to-issue date gap",
        "Evidence": f"{pricing_to_issue_days} calendar days",
        "Status": "CHECK" if pricing_to_issue_days > 10 else "PASS",
        "Use in Day 2": "Not used; confirm before note pricing",
    },
    {
        "Dataset / check": "Official note initial references",
        "Evidence": f"{np.count_nonzero(inputs['initial_references'] > 0)}/3 positive",
        "Status": "PASS" if (inputs["initial_references"] > 0).all() else "BLOCKED",
        "Use in Day 2": "Issuer_Initial_Value supplement; frozen for later note pricing",
    },
    {
        "Dataset / check": "Valid historical note prices",
        "Evidence": f"{inputs['valid_note_prices']} positive observations",
        "Status": "NOT REQUIRED",
        "Use in Day 2": "Later live-note market validation only",
    },
    {
        "Dataset / check": "Valid listed-option quotes",
        "Evidence": f"{inputs['valid_listed_options']} two-sided contracts",
        "Status": "NOT REQUIRED",
        "Use in Day 2": "Analytic Black-Scholes validation used",
    },
]
quality_table = pd.DataFrame(quality_rows)
display(quality_table)

assert (inputs["spots"] > 0).all()
assert (inputs["dividend_yields"] >= 0).all()
assert (inputs["volatilities"] > 0).all()
assert inputs["risk_free_rate"] > 0
assert (inputs["initial_references"] > 0).all()
assert len(inputs["prices"]) >= 200
assert eigenvalues.min() > -1e-10

print(
    "DAY 2 DATA GATE PASSED: contractual initial references and model inputs "
    "are available. Missing secondary note prices only limit later market validation."
)

### 2. Black–Scholes benchmark

\[
C=S_0e^{-qT}\Phi(d_1)-Ke^{-rT}\Phi(d_2),
\]

\[
\Delta=e^{-qT}\Phi(d_1),\qquad
\text{Vega}=S_0e^{-qT}\phi(d_1)\sqrt T,\qquad
\Gamma=\frac{e^{-qT}\phi(d_1)}{S_0\sigma\sqrt T}.
\]

In [ ]:
def black_scholes_call_and_greeks(s0, strike, r, q, sigma, maturity):
    vol_t = sigma * np.sqrt(maturity)
    d1 = (
        np.log(s0 / strike)
        + (r - q + 0.5 * sigma**2) * maturity
    ) / vol_t
    d2 = d1 - vol_t
    discount_q = np.exp(-q * maturity)
    discount_r = np.exp(-r * maturity)
    price = s0 * discount_q * norm.cdf(d1) - strike * discount_r * norm.cdf(d2)
    delta = discount_q * norm.cdf(d1)
    vega = s0 * discount_q * norm.pdf(d1) * np.sqrt(maturity)
    gamma = discount_q * norm.pdf(d1) / (s0 * sigma * np.sqrt(maturity))
    return {"price": price, "delta": delta, "vega": vega, "gamma": gamma}


strikes = inputs["spots"].copy()
analytic = black_scholes_call_and_greeks(
    inputs["spots"],
    strikes,
    inputs["risk_free_rate"],
    inputs["dividend_yields"],
    inputs["volatilities"],
    T,
)
analytic_table = pd.DataFrame(
    {
        "Ticker": inputs["tickers"],
        "Spot / strike": inputs["spots"],
        "r": inputs["risk_free_rate"],
        "q": inputs["dividend_yields"],
        "sigma": inputs["volatilities"],
        "Price": analytic["price"],
        "Delta": analytic["delta"],
        "Vega per 1.00 sigma": analytic["vega"],
        "Vega per vol point": analytic["vega"] / 100.0,
        "Gamma": analytic["gamma"],
    }
)
display(analytic_table)

## Results

### 3. Unified MC/RQMC sampling interface

The Sobol dimension is `N_STEPS × 3`. Uniform coordinates are generated in time-major, asset-minor order, clipped to \((\varepsilon,1-\varepsilon)\), transformed to normals, and correlated with the same Cholesky matrix used by MC.

In [ ]:
def make_psd_correlation(correlation, floor=1e-12):
    correlation = np.asarray(correlation, dtype=float)
    correlation = 0.5 * (correlation + correlation.T)
    values, vectors = np.linalg.eigh(correlation)
    values = np.maximum(values, floor)
    repaired = (vectors * values) @ vectors.T
    scale = np.sqrt(np.diag(repaired))
    repaired = repaired / np.outer(scale, scale)
    np.fill_diagonal(repaired, 1.0)
    return repaired


CORRELATION = make_psd_correlation(inputs["correlation"])
CHOLESKY = np.linalg.cholesky(CORRELATION)


def is_power_of_two(value):
    return value > 0 and (value & (value - 1)) == 0


def generate_brownian_terminal(method, sample_size, seed):
    if not is_power_of_two(sample_size):
        raise ValueError("sample_size must be a power of two")
    dimension = N_STEPS * len(inputs["tickers"])

    if method == "MC":
        rng = np.random.default_rng(seed)
        independent_normals = rng.standard_normal(
            (sample_size, N_STEPS, len(inputs["tickers"]))
        )
    elif method == "RQMC":
        sobol = qmc.Sobol(d=dimension, scramble=True, seed=seed)
        uniforms = sobol.random_base2(m=int(np.log2(sample_size)))
        uniforms = np.clip(uniforms, UNIFORM_EPS, 1.0 - UNIFORM_EPS)
        # C-order reshape means asset is the fastest-moving coordinate:
        # (time 0, asset 0), (time 0, asset 1), ...
        independent_normals = norm.ppf(uniforms).reshape(
            sample_size, N_STEPS, len(inputs["tickers"])
        )
    else:
        raise ValueError("method must be 'MC' or 'RQMC'")

    correlated_normals = independent_normals @ CHOLESKY.T
    dt = T / N_STEPS
    return np.sqrt(dt) * correlated_normals.sum(axis=1)


def terminal_levels(s0, r, q, sigma, maturity, brownian_terminal):
    return s0 * np.exp(
        (r - q - 0.5 * sigma**2) * maturity
        + sigma * brownian_terminal
    )


def estimate_price_and_greeks(brownian_terminal, gamma_bump=GAMMA_BUMP):
    s0 = inputs["spots"]
    q = inputs["dividend_yields"]
    sigma = inputs["volatilities"]
    r = inputs["risk_free_rate"]
    discount = np.exp(-r * T)

    base_terminal = terminal_levels(s0, r, q, sigma, T, brownian_terminal)
    base_payoff = discount * np.maximum(base_terminal - strikes, 0.0)

    delta_h = DELTA_BUMP * s0
    delta_up = terminal_levels(s0 + delta_h, r, q, sigma, T, brownian_terminal)
    delta_down = terminal_levels(s0 - delta_h, r, q, sigma, T, brownian_terminal)
    delta_path = discount * (
        np.maximum(delta_up - strikes, 0.0)
        - np.maximum(delta_down - strikes, 0.0)
    ) / (2.0 * delta_h)

    sigma_up = sigma + VEGA_BUMP_ABS
    sigma_down = sigma - VEGA_BUMP_ABS
    vega_up = terminal_levels(s0, r, q, sigma_up, T, brownian_terminal)
    vega_down = terminal_levels(s0, r, q, sigma_down, T, brownian_terminal)
    vega_path = discount * (
        np.maximum(vega_up - strikes, 0.0)
        - np.maximum(vega_down - strikes, 0.0)
    ) / (2.0 * VEGA_BUMP_ABS)

    gamma_h = gamma_bump * s0
    gamma_up = terminal_levels(s0 + gamma_h, r, q, sigma, T, brownian_terminal)
    gamma_down = terminal_levels(s0 - gamma_h, r, q, sigma, T, brownian_terminal)
    gamma_path = discount * (
        np.maximum(gamma_up - strikes, 0.0)
        - 2.0 * np.maximum(base_terminal - strikes, 0.0)
        + np.maximum(gamma_down - strikes, 0.0)
    ) / (gamma_h**2)

    return {
        "price": base_payoff.mean(axis=0),
        "delta": delta_path.mean(axis=0),
        "vega": vega_path.mean(axis=0),
        "gamma": gamma_path.mean(axis=0),
    }


# Marginal distribution and correlation smoke test at the largest N.
smoke = {}
for method in ("MC", "RQMC"):
    brownian = generate_brownian_terminal(method, SAMPLE_SIZES[-1], BASE_SEED)
    smoke[method] = {
        "Mean": brownian.mean(axis=0),
        "Variance": brownian.var(axis=0, ddof=1),
        "Max correlation error": np.max(
            np.abs(np.corrcoef(brownian, rowvar=False) - CORRELATION)
        ),
    }
smoke_table = pd.concat(
    {
        method: pd.DataFrame(
            {
                "Ticker": inputs["tickers"],
                "Brownian mean": values["Mean"],
                "Brownian variance": values["Variance"],
                "Max correlation error": values["Max correlation error"],
            }
        )
        for method, values in smoke.items()
    },
    names=["Method", "Row"],
).reset_index(level="Row", drop=True)
display(smoke_table)

In [ ]:
def run_replication(method, sample_size, replication):
    seed = (
        BASE_SEED
        + (0 if method == "MC" else 10_000_000)
        + 100_000 * int(np.log2(sample_size))
        + replication
    )
    start = time.perf_counter()
    brownian = generate_brownian_terminal(method, sample_size, seed)
    estimates = estimate_price_and_greeks(brownian)
    elapsed = time.perf_counter() - start

    rows = []
    for asset_index, ticker in enumerate(inputs["tickers"]):
        for metric in ("price", "delta", "vega", "gamma"):
            rows.append(
                {
                    "Method": method,
                    "N": sample_size,
                    "Replication": replication,
                    "Seed / scramble": seed,
                    "Ticker": ticker,
                    "Metric": metric,
                    "Estimate": estimates[metric][asset_index],
                    "Analytic": analytic[metric][asset_index],
                    "Runtime seconds": elapsed,
                }
            )
    return rows


raw_rows = []
experiment_start = time.perf_counter()
for method in ("MC", "RQMC"):
    for sample_size in SAMPLE_SIZES:
        for replication in range(N_REPLICATIONS):
            raw_rows.extend(run_replication(method, sample_size, replication))
raw_results = pd.DataFrame(raw_rows)
experiment_runtime = time.perf_counter() - experiment_start
raw_results["Error"] = raw_results["Estimate"] - raw_results["Analytic"]
raw_results["Relative error"] = raw_results["Error"] / raw_results["Analytic"].abs()

summary = (
    raw_results.groupby(["Method", "N", "Ticker", "Metric"], as_index=False)
    .agg(
        Mean_estimate=("Estimate", "mean"),
        Analytic=("Analytic", "first"),
        Replication_SD=("Estimate", "std"),
        Mean_runtime_seconds=("Runtime seconds", "mean"),
    )
)
rmse = (
    raw_results.assign(Squared_error=raw_results["Error"] ** 2)
    .groupby(["Method", "N", "Ticker", "Metric"], as_index=False)
    .agg(RMSE=("Squared_error", lambda values: np.sqrt(values.mean())))
)
summary = summary.merge(rmse, on=["Method", "N", "Ticker", "Metric"])
summary["Bias"] = summary["Mean_estimate"] - summary["Analytic"]
summary["Relative_RMSE"] = summary["RMSE"] / summary["Analytic"].abs()
summary["SE_of_replication_mean"] = summary["Replication_SD"] / np.sqrt(N_REPLICATIONS)

print(f"Experiment runtime: {experiment_runtime:.1f} seconds")
display(raw_results.head(8))

### 4. Convergence results

The primary error measure is replication RMSE against the Black–Scholes benchmark. The RQMC RMSE is computed across independent scrambles.

In [ ]:
largest_n = SAMPLE_SIZES[-1]
largest_n_results = summary[summary["N"] == largest_n].copy()
largest_n_results["z score of mean"] = (
    largest_n_results["Bias"] / largest_n_results["SE_of_replication_mean"]
)
display(
    largest_n_results[
        [
            "Method", "Ticker", "Metric", "Mean_estimate", "Analytic",
            "Bias", "RMSE", "Replication_SD", "z score of mean",
            "Mean_runtime_seconds",
        ]
    ].sort_values(["Metric", "Ticker", "Method"])
)

aggregate_convergence = (
    summary.groupby(["Method", "N", "Metric"], as_index=False)
    .agg(
        Mean_relative_RMSE=("Relative_RMSE", "mean"),
        Median_runtime_seconds=("Mean_runtime_seconds", "median"),
    )
)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
for axis, metric in zip(axes.flat, ("price", "delta", "vega", "gamma")):
    metric_data = aggregate_convergence[aggregate_convergence["Metric"] == metric]
    for method, marker in (("MC", "o"), ("RQMC", "s")):
        subset = metric_data[metric_data["Method"] == method]
        axis.loglog(
            subset["N"], subset["Mean_relative_RMSE"],
            marker=marker, linewidth=2, label=method
        )
    axis.set(
        title=f"{metric.title()} relative RMSE",
        xlabel="Paths per replication (N)",
        ylabel="Mean relative RMSE across 3 assets",
    )
    axis.legend()
plt.suptitle("European price and Greek convergence: MC vs scrambled Sobol RQMC")
plt.tight_layout()
plt.show()

In [ ]:
def convergence_slope(group):
    x = np.log(group["N"].to_numpy(dtype=float))
    y = np.log(group["Mean_relative_RMSE"].to_numpy(dtype=float))
    return np.polyfit(x, y, 1)[0]


slopes = (
    aggregate_convergence.groupby(["Method", "Metric"])
    .apply(convergence_slope, include_groups=False)
    .rename("Empirical log-log slope")
    .reset_index()
)
display(slopes)

comparison = aggregate_convergence.pivot(
    index=["N", "Metric"], columns="Method", values="Mean_relative_RMSE"
).reset_index()
comparison["RQMC / MC RMSE"] = comparison["RQMC"] / comparison["MC"]
display(comparison.sort_values(["Metric", "N"]))

largest_aggregate = aggregate_convergence[
    aggregate_convergence["N"] == largest_n
].copy()
rmse_tolerances = {
    "price": 0.02,
    "delta": 0.02,
    "vega": 0.02,
    "gamma": 0.05,
}
largest_aggregate["Tolerance"] = largest_aggregate["Metric"].map(rmse_tolerances)
largest_aggregate["Status"] = np.where(
    largest_aggregate["Mean_relative_RMSE"] <= largest_aggregate["Tolerance"],
    "PASS",
    "CHECK",
)
validation_checks = pd.DataFrame(
    [
        {
            "Check": "All largest-N relative RMSE thresholds",
            "Actual": largest_aggregate["Status"].eq("PASS").all(),
            "Threshold": "Price/Delta/Vega <=2%; Gamma <=5%",
            "Status": "PASS"
            if largest_aggregate["Status"].eq("PASS").all()
            else "CHECK",
        },
        {
            "Check": "Largest-N replication-mean z scores",
            "Actual": largest_n_results["z score of mean"].abs().max(),
            "Threshold": "< 3.5",
            "Status": "PASS"
            if largest_n_results["z score of mean"].abs().max() < 3.5
            else "CHECK",
        },
        {
            "Check": "RQMC improves price, Delta and Vega RMSE",
            "Actual": comparison[
                (comparison["N"] == largest_n)
                & comparison["Metric"].isin(["price", "delta", "vega"])
            ]["RQMC / MC RMSE"].max(),
            "Threshold": "< 1.0",
            "Status": "PASS"
            if comparison[
                (comparison["N"] == largest_n)
                & comparison["Metric"].isin(["price", "delta", "vega"])
            ]["RQMC / MC RMSE"].max() < 1.0
            else "CHECK",
        },
    ]
)
display(largest_aggregate)
display(validation_checks)

assert largest_aggregate["Status"].eq("PASS").all()
assert largest_n_results["z score of mean"].abs().max() < 3.5
assert validation_checks["Status"].eq("PASS").all()

### 5. Gamma bump-size stability

Gamma is a second difference and therefore exposes both sampling noise and finite-difference bias. The same Brownian terminal draws are reused across all bump sizes within each replication.

In [ ]:
GAMMA_BUMPS = [0.0025, 0.005, 0.01, 0.02]
gamma_rows = []
gamma_sample_size = SAMPLE_SIZES[-1]

for method in ("MC", "RQMC"):
    for replication in range(N_REPLICATIONS):
        seed = (
            BASE_SEED
            + (20_000_000 if method == "MC" else 30_000_000)
            + replication
        )
        brownian = generate_brownian_terminal(method, gamma_sample_size, seed)
        for bump in GAMMA_BUMPS:
            gamma_estimate = estimate_price_and_greeks(
                brownian, gamma_bump=bump
            )["gamma"]
            for asset_index, ticker in enumerate(inputs["tickers"]):
                gamma_rows.append(
                    {
                        "Method": method,
                        "Replication": replication,
                        "Ticker": ticker,
                        "Bump": bump,
                        "Gamma": gamma_estimate[asset_index],
                        "Analytic": analytic["gamma"][asset_index],
                    }
                )

gamma_raw = pd.DataFrame(gamma_rows)
gamma_raw["Error"] = gamma_raw["Gamma"] - gamma_raw["Analytic"]
gamma_summary = (
    gamma_raw.groupby(["Method", "Ticker", "Bump"], as_index=False)
    .agg(
        Mean_Gamma=("Gamma", "mean"),
        Analytic=("Analytic", "first"),
        Replication_SD=("Gamma", "std"),
        RMSE=("Error", lambda values: np.sqrt(np.mean(values**2))),
    )
)
gamma_summary["Estimate / analytic"] = (
    gamma_summary["Mean_Gamma"] / gamma_summary["Analytic"]
)
display(gamma_summary)

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for axis, ticker in zip(axes, inputs["tickers"]):
    subset = gamma_summary[gamma_summary["Ticker"] == ticker]
    for method, marker in (("MC", "o"), ("RQMC", "s")):
        method_data = subset[subset["Method"] == method]
        axis.errorbar(
            100 * method_data["Bump"],
            method_data["Estimate / analytic"],
            yerr=method_data["Replication_SD"] / method_data["Analytic"],
            marker=marker,
            capsize=3,
            label=method,
        )
    axis.axhline(1.0, color="black", linestyle="--", linewidth=1)
    axis.set(
        title=ticker,
        xlabel="Spot bump (%)",
        ylabel="Estimated / analytic Gamma",
    )
    axis.legend()
plt.suptitle(f"Gamma bump stability at N={gamma_sample_size:,}")
plt.tight_layout()
plt.show()

### 6. Save auditable experiment outputs

Replication-level estimates are saved separately from summary tables so later report figures can be rebuilt without rerunning the full simulation.

In [ ]:
raw_results.to_csv(OUTPUT_DIR / "replication_results.csv", index=False)
summary.to_csv(OUTPUT_DIR / "convergence_summary.csv", index=False)
gamma_summary.to_csv(OUTPUT_DIR / "gamma_bump_summary.csv", index=False)
quality_table.to_csv(OUTPUT_DIR / "data_quality_summary.csv", index=False)

run_manifest = pd.DataFrame(
    [
        {
            "Project-relative source": SOURCE_FILE.relative_to(PROJECT_DIR).as_posix(),
            "Source SHA-256": SOURCE_SHA256,
            "Frozen config": CONFIG_FILE.relative_to(PROJECT_DIR).as_posix(),
            "Bloomberg ID": inputs["metadata"]["Bloomberg ID"],
            "As-of date": inputs["metadata"]["As-of date"],
            "Model": "Risk-neutral correlated GBM",
            "Methods": "MC; independently scrambled Sobol RQMC",
            "Sample sizes": ", ".join(map(str, SAMPLE_SIZES)),
            "Replications / scrambles": N_REPLICATIONS,
            "Time steps": N_STEPS,
            "Coordinate order": "time-major, asset-minor",
            "Uniform clipping epsilon": UNIFORM_EPS,
            "Delta bump": DELTA_BUMP,
            "Vega absolute sigma bump": VEGA_BUMP_ABS,
            "Gamma spot bumps": ", ".join(map(str, GAMMA_BUMPS)),
            "Base seed": BASE_SEED,
        }
    ]
)
run_manifest.to_csv(OUTPUT_DIR / "run_manifest.csv", index=False)

print(f"Saved model outputs to: {OUTPUT_DIR}")
print(sorted(path.name for path in OUTPUT_DIR.glob("*.csv")))

## Takeaways

The executed tables and charts above support the following interpretation:

1. The correlated multi-step engine preserves the required marginal Brownian moments and correlation structure under both MC and RQMC.
2. Price, Delta, Vega and Gamma are benchmarked against closed forms using the same production sampling interface.
3. RQMC performance must be judged from independent scrambles and may differ by metric; a method that improves price need not improve second-order Gamma by the same factor.
4. Gamma stability depends on both \(N\) and bump size. The bump study should be reviewed before the autocallable Gamma experiment is frozen.
5. The HSBC workbook and issuer supplement are sufficient for Day 2 and freeze all three contractual initial references. Genuine secondary note quotes and listed-option contracts remain unavailable, which limits later market validation but not the present engine test.

### Limitations

- The 3Y SOFR series is a proxy for a one-year validation option.
- The validation uses 12M ATM vol rather than a strike/expiry volatility surface.
- Analytic Black–Scholes validation establishes implementation correctness under GBM; it does not establish market-model adequacy.
- The note-price zeros are rejected as sentinels and are not evidence about the HSBC security's market value.